# FOLLOWING COMMANDS TO BE RUN IN NEPER
# FOR GENERATING 100 GRAIN PRISMATIC VTM OF 0.02 x 0.02 x 0.05 m^3 DIMENSION WITH RANDOM GRAIN SIZE DISTRIBUTION
neper -T -n 100 -domain "cube(0.02,0.02,0.05)" -format tess,geo -o prismatic100
# TO VIEW THE SPECIMEN
neper -V prismatic100.tess -datacellcol id -datacelltrs 0.5 -print prismatic100_image
# AFTER GENERATING .TESS AND .GEO, THE NEXT STEP INCLUDES EXECUTION OF THE FOLLOWING PYTHON CODE FOR GENERATING .STL FILE

In [ ]:
import re
import numpy as np
from collections import defaultdict

# ── CONFIGURE THESE PATHS ──────────────────────────────────────────────────────
geo_file = input("Enter path to Neper .geo file: ").strip().strip('"').strip("'") # # your Neper .geo file
combined_stl = input("Enter path for output STL file (grains + joints): ").strip().strip('"').strip("'") ## ONE output file: grains + joints

if not combined_stl.lower().endswith(".stl"):
    combined_stl += ".stl"
# ──────────────────────────────────────────────────────────────────────────────

# =============================================================================
# 1. PARSE THE .GEO FILE
# =============================================================================
with open(geo_file, "r") as f:
    lines = f.readlines()

points         = {}
lines_dict     = {}
line_loops     = {}
plane_surfaces = {}
surface_loops  = {}
volumes        = {}

for line in lines:
    line = line.strip()

    m = re.match(r'Point\s*\((\d+)\)\s*=\s*\{([\d\.\-eE\+\s,]+)\}', line)
    if m:
        pid  = int(m.group(1))
        vals = [float(x.strip()) for x in m.group(2).split(',') if x.strip()]
        if len(vals) >= 3:
            points[pid] = np.array(vals[:3])
        continue

    m = re.match(r'(?<!\w)Line\s*\((\d+)\)\s*=\s*\{([^}]+)\}', line)
    if m:
        lid = int(m.group(1))
        pts = [int(x.strip()) for x in m.group(2).split(',') if x.strip()]
        lines_dict[lid] = pts
        continue

    m = re.match(r'Line Loop\s*\((\d+)\)\s*=\s*\{([^}]+)\}', line)
    if m:
        llid = int(m.group(1))
        lids = [int(x.strip()) for x in m.group(2).split(',') if x.strip()]
        line_loops[llid] = lids
        continue

    m = re.match(r'Plane Surface\s*\((\d+)\)\s*=\s*\{([^}]+)\}', line)
    if m:
        sid   = int(m.group(1))
        loops = [int(x.strip()) for x in m.group(2).split(',') if x.strip()]
        plane_surfaces[sid] = loops
        continue

    m = re.match(r'Surface Loop\s*\((\d+)\)\s*=\s*\{([^}]+)\}', line)
    if m:
        slid = int(m.group(1))
        sids = [int(x.strip()) for x in m.group(2).split(',') if x.strip()]
        surface_loops[slid] = sids
        continue

    m = re.match(r'(?<!\w)Volume\s*\((\d+)\)\s*=\s*\{([^}]+)\}', line)
    if m:
        vid   = int(m.group(1))
        slids = [int(x.strip()) for x in m.group(2).split(',') if x.strip()]
        volumes[vid] = slids
        continue

print(f"Parsed — Points: {len(points)}, Lines: {len(lines_dict)}, "
      f"Plane Surfaces: {len(plane_surfaces)}, Volumes: {len(volumes)}")

# =============================================================================
# 2. FIND SHARED SURFACES (+ sanity check: no face should be shared by 3+ grains)
# =============================================================================
surface_to_volumes = defaultdict(set)

for vol_id, slids in volumes.items():
    for slid in slids:
        sl = abs(slid)
        if sl not in surface_loops:
            continue
        for sid in surface_loops[sl]:
            surface_to_volumes[abs(sid)].add(vol_id)

shared_info = {}
bad_surfaces = {}
for sid, vols in surface_to_volumes.items():
    if len(vols) == 2:
        sorted_vols = sorted(vols)
        shared_info[sid] = (sorted_vols[0], sorted_vols[1])
    elif len(vols) >= 3:
        # Should never happen for a Voronoi/Laguerre tessellation (a face is by
        # construction the bisector between exactly two seeds). If this fires,
        # something is off upstream (degenerate geometry, duplicate IDs, etc.)
        bad_surfaces[sid] = sorted(vols)

print(f"Total surfaces      : {len(plane_surfaces)}")
print(f"Shared (joint) faces: {len(shared_info)}")
print(f"Boundary faces      : {len(plane_surfaces) - len(shared_info) - len(bad_surfaces)}")

if bad_surfaces:
    print(f"\n⚠ WARNING: {len(bad_surfaces)} surface(s) referenced by 3+ volumes — "
          f"this should be topologically impossible. Inspect these before trusting the output:")
    for sid, vols in list(bad_surfaces.items())[:10]:
        print(f"   surface {sid}: volumes {vols}")
else:
    print("Sanity check passed: no surface is shared by more than 2 volumes.")

# =============================================================================
# 3. HELPER FUNCTIONS
# =============================================================================
def get_face_vertices(surface_id):
    sid = abs(surface_id)
    if sid not in plane_surfaces:
        return []
    loop_id = abs(plane_surfaces[sid][0])
    if loop_id not in line_loops:
        return []
    line_ids = line_loops[loop_id]
    verts = []
    for lid in line_ids:
        if abs(lid) not in lines_dict:
            continue
        l = lines_dict[abs(lid)]
        verts.append(l[0] if lid > 0 else l[-1])
    return verts

def triangulate_polygon(vids, flip=False):
    if len(vids) < 3:
        return []
    tris = []
    v0 = vids[0]
    for i in range(1, len(vids) - 1):
        if flip:
            tris.append((v0, vids[i+1], vids[i]))
        else:
            tris.append((v0, vids[i], vids[i+1]))
    return tris

def compute_normal(p1, p2, p3):
    v1 = p2 - p1
    v2 = p3 - p1
    n  = np.cross(v1, v2)
    norm = np.linalg.norm(n)
    return n / norm if norm > 0 else np.array([0.0, 0.0, 1.0])

def face_vertex_coords(surface_id):
    vids = get_face_vertices(surface_id)
    return [points[v] for v in vids if v in points]

def grain_centroid(grain_id):
    """Average of all vertices referenced by this grain's bounding surfaces."""
    pts_seen = set()
    for slid in volumes[grain_id]:
        sl = abs(slid)
        for sid in surface_loops.get(sl, []):
            for pid in get_face_vertices(sid):
                pts_seen.add(pid)
    coords = np.array([points[p] for p in pts_seen if p in points])
    return coords.mean(axis=0)

def write_solid(f, solid_name, surface_ids_with_flip):
    """Write one solid block containing all triangles for given surfaces."""
    facet_count = 0
    tris_buffer = []

    for (sid, flip) in surface_ids_with_flip:
        vids = get_face_vertices(sid)
        if len(vids) < 3:
            continue
        tris = triangulate_polygon(vids, flip=flip)
        for tri in tris:
            try:
                p1 = points[tri[0]]
                p2 = points[tri[1]]
                p3 = points[tri[2]]
                n  = compute_normal(p1, p2, p3)
                tris_buffer.append((n, p1, p2, p3))
                facet_count += 1
            except Exception as e:
                print(f"  Skipped tri in surface {sid}: {e}")

    if not tris_buffer:
        return 0

    f.write(f"solid {solid_name}\n")
    for (n, p1, p2, p3) in tris_buffer:
        f.write(f"  facet normal {n[0]:.6e} {n[1]:.6e} {n[2]:.6e}\n")
        f.write(f"    outer loop\n")
        f.write(f"      vertex {p1[0]:.6e} {p1[1]:.6e} {p1[2]:.6e}\n")
        f.write(f"      vertex {p2[0]:.6e} {p2[1]:.6e} {p2[2]:.6e}\n")
        f.write(f"      vertex {p3[0]:.6e} {p3[1]:.6e} {p3[2]:.6e}\n")
        f.write(f"    endloop\n")
        f.write(f"  endfacet\n")
    f.write(f"endsolid {solid_name}\n\n")
    return facet_count

# =============================================================================
# 4. PRECOMPUTE GRAIN CENTROIDS (needed for consistent joint normal direction)
# =============================================================================
print("\nComputing grain centroids for joint normal orientation...")
centroids = {vid: grain_centroid(vid) for vid in volumes}

def joint_flip(sid, grain_a, grain_b):
    """
    Decide whether to flip this joint face so its normal consistently points
    from grain_a's centroid toward grain_b's centroid (fixed convention,
    same rule for every joint — needed for RS3 dip/dip-direction consistency).
    """
    coords = face_vertex_coords(sid)
    if len(coords) < 3:
        return False
    face_center = np.mean(coords, axis=0)
    n_unflipped = compute_normal(coords[0], coords[1], coords[2])
    to_b = centroids[grain_b] - face_center
    pointing_a_to_b = np.dot(n_unflipped, to_b) > 0
    # if it's already pointing a->b, no flip needed; otherwise flip it
    return not pointing_a_to_b

# =============================================================================
# 5. WRITE COMBINED STL
#    PART A — Grains: solid poly{grain_id}
#    PART B — Joints: solid joint_surf{id}_grain{a}_grain{b}, normal a→b always
# =============================================================================
print(f"\nWriting combined STL → {combined_stl}")

grain_solids  = 0
grain_facets  = 0
joint_solids  = 0
joint_facets  = 0

with open(combined_stl, "w") as f:

    # ── PART A: GRAINS ────────────────────────────────────────────────────────
    for grain_id, slids in sorted(volumes.items()):
        surfaces_for_grain = []
        for slid in slids:
            sl = abs(slid)
            if sl not in surface_loops:
                continue
            for sid in surface_loops[sl]:
                flip = sid < 0
                surfaces_for_grain.append((abs(sid), flip))

        solid_name = f"poly{grain_id}"
        count = write_solid(f, solid_name, surfaces_for_grain)
        if count > 0:
            grain_solids += 1
            grain_facets += count

    # ── PART B: JOINT SURFACES (consistent a→b normal convention) ───────────────
    for sid in sorted(shared_info.keys()):
        grain_a, grain_b = shared_info[sid]
        flip = joint_flip(sid, grain_a, grain_b)
        solid_name = f"joint_surf{sid}_grain{grain_a}_grain{grain_b}"
        count = write_solid(f, solid_name, [(sid, flip)])
        if count > 0:
            joint_solids += 1
            joint_facets += count

print(f"\nDone!")
print(f"  Grain solids  : {grain_solids}  ({grain_facets} facets)")
print(f"  Joint solids  : {joint_solids}  ({joint_facets} facets)")
print(f"  Total solids  : {grain_solids + joint_solids}")
print(f"\nEvery joint normal now points consistently from the lower-numbered")
print(f"grain toward the higher-numbered grain (grain_a → grain_b).")
print(f"\nIn RS3:")
print(f"  • Import '{combined_stl}' once")
print(f"  • Grains show as:  poly1, poly2, ... → assign as Geology volumes")
print(f"  • Joints show as:  joint_surf<id>_grain<A>_grain<B> → select & add joint surface")

# FOR GENERATING GRAIN SIZE AND VOLUME DISTRIBUTION HISTOGRAM
# FOLLOWING COMMANDS TO BE RUN IN NEPER
neper -T -loadtess prismatic100.tess -statcell vol -o prismatic100_grain_volume #FOR GRAIN VOLUME DATA
neper -T -loadtess prismatic100.tess -statcell diameq -o prismatic100_grain_diameq #FOR GRAIN DIAMETER EQUIVALENT DATA
# AFTER GENERATING STCELL FILES THE NEXT STEP INCLUDES EXECUTION OF THE FOLLOWING PYTHON CODE FOR GENERATING GRAIN SIZE AND VOLUME DISTRIBUTION HISTOGRAM


In [ ]:
#FOR GRAIN VOLUME DISTRIBUTION
import numpy as np
import matplotlib.pyplot as plt

# Ask user for the Neper .stcell file path
file_path = input("Enter path to the volume .stcell file: ").strip('"')

# Load volumes (Neper outputs m³)
volumes = np.loadtxt(file_path)

# Convert m³ -> mm³
volumes = volumes * 1e9

plt.figure(figsize=(12,6))

plt.hist(
    volumes,
    bins=20,              # more bars
    edgecolor='black',
    linewidth=1
)

# Tick spacing
xmin = np.floor(volumes.min()*10)/10
xmax = np.ceil(volumes.max()*10)/10

plt.xticks(
    np.arange(xmin, xmax, 20)
)

plt.xlabel(r'Grain Volume (mm$^3$)', fontsize=12)
plt.ylabel('Number of Grains', fontsize=12)
plt.title('Grain Volume Distribution', fontsize=13)

plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
#FOR GRAIN SIZE DISTRIBUTION
import numpy as np
import matplotlib.pyplot as plt

# Ask user for the Neper .stcell file path
file_path = input("Enter path to the diameq .stcell file: ").strip('"')

# Load equivalent grain diameters from Neper
diam = np.loadtxt(file_path)

# Convert m -> mm
diam = diam * 1000

plt.figure(figsize=(10,6))

plt.hist(
    diam,
    bins=20,
    edgecolor='black',
    linewidth=1
)

# Mean grain size line
plt.axvline(
    np.mean(diam),
    linestyle='--',
    linewidth=2,
    label=f'Mean = {np.mean(diam):.2f} mm'
)

# Tick spacing (adjust if needed)
xmin = np.floor(diam.min()*10)/10
xmax = np.ceil(diam.max()*10)/10

plt.xticks(
    np.arange(xmin, xmax+0.2, 0.2),
    rotation=45
)

plt.xlabel('Equivalent Grain Diameter (mm)', fontsize=12)
plt.ylabel('Number of Grains', fontsize=12)
plt.title('Grain Size Distribution', fontsize=14)

plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()